# XLM-R — One-to-Many (OTM), Unweighted BCE

Trains XLM-R on the English subset of MultiEURLEX (OTM configuration) and evaluates
the resulting model on all 23 languages of the test set.

In [ ]:
!pip -q install -U datasets transformers accelerate scikit-learn psutil

from collections import Counter
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/multieurlex_all_languages")

label_feature = dataset["train"].features["labels"].feature
print("Train / Val / Test sizes:",
      dataset["train"].num_rows, dataset["validation"].num_rows, dataset["test"].num_rows)
print("Total possible label classes:", len(label_feature.names))

## Configuration

In [ ]:
import os, time, random, gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from accelerate import Accelerator

# fp32 is used throughout (bf16 was found to collapse under unweighted BCE
# with this label distribution)
accelerator = Accelerator(mixed_precision='no')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
THRESHOLD = 0.50
N_TRAIN_PARTS = 10          # training data is processed in chunks to limit RAM use

#All three transformer models (to be ran one at a time).
MODEL_NAME = "bert-base-uncased"               # BERT
#MODEL_NAME = "bert-base-multilingual-cased"   # mBERT
#MODEL_NAME = "xlm-roberta-base"               # XLM-R

TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"

total_training_samples = len(dataset["train"])
total_steps = (total_training_samples // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

print("Model:", MODEL_NAME)
print("Total training samples:", total_training_samples)
print("Total training steps:", total_steps)
print("Warmup steps:", warmup_steps)

## Metrics

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def multilabel_metrics(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "exact_match": accuracy_score(y_true, y_pred),
    }

## Label vocabulary

In [ ]:
all_label_names = label_feature.names
NUM_LABELS = len(all_label_names)
label_to_index = {i: i for i in range(NUM_LABELS)}

print("Number of labels:", NUM_LABELS)

def multi_hot(label_ids, mapping):
    vector = np.zeros(len(mapping), dtype=np.float32)
    for label_id in label_ids:
        vector[mapping[int(label_id)]] = 1.0
    return vector

## Preprocessing (English-only, OTM)

OTM trains and validates on the English version of each document.

In [ ]:
from functools import partial
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples, tokenizer, label_to_index, max_length):
    texts = [example.get('en', '') for example in examples[TEXT_COLUMN]]
    tokenized = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
    tokenized["labels"] = [
        multi_hot([lbl for lbl in row if int(lbl) in label_to_index], label_to_index)
        for row in examples[LABEL_COLUMN]
    ]
    return tokenized

preprocess_map_fn = partial(preprocess_function, tokenizer=tokenizer,
                             label_to_index=label_to_index, max_length=MAX_LENGTH)

train_split_size = len(dataset["train"])
part_size = train_split_size // N_TRAIN_PARTS

tokenized_val = dataset["validation"].map(
    preprocess_map_fn, batched=True,
    remove_columns=[TEXT_COLUMN, LABEL_COLUMN], load_from_cache_file=False
)

print("Tokenized validation size:", len(tokenized_val))

## DataLoader

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

val_dataset_for_loader = tokenized_val.remove_columns(["celex_id"])

val_loader = DataLoader(
    val_dataset_for_loader,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

print("Validation DataLoader created.")

## Model, optimizer, scheduler

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

config = AutoConfig.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification"
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

model, optimizer, val_loader, scheduler, data_collator = accelerator.prepare(
    model, optimizer, val_loader, scheduler, data_collator
)

print("Model:", MODEL_NAME, "| Number of labels:", model.config.num_labels)

## Evaluation function (English validation set)

In [ ]:
from tqdm.auto import tqdm
from scipy.special import expit

def evaluate_model(model, loader, threshold=0.5):
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.eval()

    all_logits, all_labels = [], []
    total_loss, n_batches = 0.0, 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch["labels"] = batch["labels"].to(torch.float32)
            outputs = unwrapped_model(**batch)

            total_loss += outputs.loss.item()
            n_batches += 1
            all_logits.append(outputs.logits.detach().cpu())
            all_labels.append(batch["labels"].detach().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probabilities = expit(logits)
    predictions = (probabilities >= threshold).astype(int)

    metrics = multilabel_metrics(labels, predictions)
    metrics["loss"] = total_loss / max(n_batches, 1)
    return metrics, probabilities, predictions, labels

## Training loop

In [ ]:
training_start = time.perf_counter()
history = []

train_split_size = len(dataset["train"])
part_size = train_split_size // N_TRAIN_PARTS

for epoch in tqdm(range(NUM_EPOCHS), desc="Epochs"):
    model.train()
    running_loss = 0.0
    current_step_in_epoch = 0
    epoch_start = time.perf_counter()

    for part_idx in range(N_TRAIN_PARTS):
        start_idx = part_idx * part_size
        end_idx = (part_idx + 1) * part_size if part_idx < N_TRAIN_PARTS - 1 else train_split_size

        raw_train_part = dataset["train"].select(range(start_idx, end_idx))
        tokenized_part = raw_train_part.map(
            preprocess_map_fn, batched=True,
            remove_columns=[TEXT_COLUMN, LABEL_COLUMN, "celex_id"],
            load_from_cache_file=False,
        )

        train_loader_part = DataLoader(
            tokenized_part, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator
        )
        prepared_train_loader_part = accelerator.prepare(train_loader_part)

        for batch in tqdm(prepared_train_loader_part,
                           desc=f"Epoch {epoch+1} part {part_idx+1}/{N_TRAIN_PARTS}", leave=False):
            optimizer.zero_grad()
            batch["labels"] = batch["labels"].to(torch.float32)

            outputs = model(**batch)
            loss = outputs.loss

            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            current_step_in_epoch += 1

        del raw_train_part, tokenized_part, train_loader_part, prepared_train_loader_part
        torch.cuda.empty_cache()
        gc.collect()

    train_loss = running_loss / max(current_step_in_epoch, 1)
    val_metrics, _, _, _ = evaluate_model(model, val_loader, threshold=THRESHOLD)
    epoch_time = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_f1": val_metrics["macro_f1"],
        "epoch_time_sec": epoch_time,
    })

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | train loss={train_loss:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | val micro-F1={val_metrics['micro_f1']:.4f} | "
        f"val macro-F1={val_metrics['macro_f1']:.4f} | time={epoch_time:.1f}s"
    )

total_training_time = time.perf_counter() - training_start
print(f"Total training time: {total_training_time/60:.2f} minutes")

## Training curves

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training vs Validation Loss"); plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["val_micro_f1"], marker="o", label="Validation micro-F1")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Validation macro-F1")
plt.xlabel("Epoch"); plt.ylabel("F1"); plt.title("Validation F1 by Epoch"); plt.legend()
plt.show()

## Multilingual evaluation (all 23 languages)

The model is trained/validated on English only (OTM); here it is evaluated on the raw test-set text for each of the 23 languages.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

LANGUAGES = ['bg', 'cs', 'da', 'de', 'el', 'en', 'es', 'et', 'fi', 'fr', 'hr',
             'hu', 'it', 'lt', 'lv', 'mt', 'nl', 'pl', 'pt', 'ro', 'sk', 'sl', 'sv']

def evaluate_multilingual_full(model, tokenizer, raw_dataset, label_to_index,
                                num_labels, languages, batch_size=16, max_length=256):
    model.eval()
    probs_by_lang, true_by_lang = {}, {}

    for lang in tqdm(languages, desc="Languages"):
        texts, label_rows = [], []
        for row in raw_dataset:
            text = row["text"].get(lang)
            if text:
                texts.append(text)
                mh = np.zeros(num_labels, dtype=np.float32)
                for l in row["labels"]:
                    if l in label_to_index:
                        mh[label_to_index[l]] = 1.0
                label_rows.append(mh)
        if not texts:
            continue

        logits_list = []
        for i in tqdm(range(0, len(texts), batch_size), desc=f"  {lang}", leave=False):
            enc = tokenizer(texts[i:i+batch_size], truncation=True, padding=True,
                             max_length=max_length, return_tensors="pt").to(accelerator.device)
            with torch.no_grad():
                out = model(**enc)
            logits_list.append(out.logits.cpu())

        probs_by_lang[lang] = torch.sigmoid(torch.cat(logits_list)).numpy()
        true_by_lang[lang] = np.stack(label_rows)

    return probs_by_lang, true_by_lang

_inf_start = time.perf_counter()
probs_by_lang, true_by_lang = evaluate_multilingual_full(
    accelerator.unwrap_model(model), tokenizer, dataset["test"],
    label_to_index, NUM_LABELS, LANGUAGES
)
inference_time_multilingual = time.perf_counter() - _inf_start

np.savez(
    "/content/drive/MyDrive/XLM-R_OTM_UNW_eval_probs_by_lang.npz",
    **{f"{l}_probs": probs_by_lang[l] for l in probs_by_lang},
    **{f"{l}_true": true_by_lang[l] for l in true_by_lang}
)
print("Saved raw probabilities + labels to Drive.")

## Inference time

In [ ]:
# Inference time, measured over the multilingual evaluation forward passes above
# (pooled_probs already holds every language's predictions from the cell above)
n_inference_examples = sum(len(p) for p in probs_by_lang.values())
print(f"Inference examples (pooled, 23 languages): {n_inference_examples}")
print(f"Total inference time: {inference_time_multilingual:.2f} sec")
print(f"Documents/sec: {n_inference_examples / inference_time_multilingual:.2f}")

## Per-language metrics, pooled and averaged overall metrics

In [ ]:
def build_per_lang_df(probs_by_lang, true_by_lang, threshold=0.5):
    results = []
    for lang, probs in probs_by_lang.items():
        preds = (probs >= threshold).astype(int)
        y_true = true_by_lang[lang]
        results.append({
            "language": lang,
            "n_examples": len(probs),
            "micro_precision": precision_score(y_true, preds, average="micro", zero_division=0),
            "micro_recall": recall_score(y_true, preds, average="micro", zero_division=0),
            "micro_f1": f1_score(y_true, preds, average="micro", zero_division=0),
            "macro_precision": precision_score(y_true, preds, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, preds, average="macro", zero_division=0),
            "macro_f1": f1_score(y_true, preds, average="macro", zero_division=0),
        })
    return pd.DataFrame(results)

per_lang_df = build_per_lang_df(probs_by_lang, true_by_lang, threshold=THRESHOLD)
print("=== Per-language metrics (threshold = 0.5) ===")
display(per_lang_df.round(4))

pooled_probs = np.concatenate(list(probs_by_lang.values()))
pooled_true = np.concatenate(list(true_by_lang.values()))
pooled_preds = (pooled_probs >= THRESHOLD).astype(int)

pooled_overall = {
    "micro_precision": precision_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "micro_recall": recall_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "micro_f1": f1_score(pooled_true, pooled_preds, average="micro", zero_division=0),
    "macro_precision": precision_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "macro_recall": recall_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "macro_f1": f1_score(pooled_true, pooled_preds, average="macro", zero_division=0),
    "exact_match": accuracy_score(pooled_true, pooled_preds),
}
print("\n=== Overall metrics — POOLED across all 23 languages ===")
display(pd.Series(pooled_overall).round(4))

overall_avg = per_lang_df[[
    "micro_precision", "micro_recall", "micro_f1",
    "macro_precision", "macro_recall", "macro_f1"
]].mean()
print("\n=== Overall metrics — AVERAGED across the 23 per-language scores ===")
display(overall_avg.round(4))

per_lang_df.to_csv("/content/drive/MyDrive/langF1(XLM-R)O-T-M_UNW.csv", index=False)
print("Saved per-language metrics to Drive.")

## Model size

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")